# LongMemEval Experiment Kaggle Pipeline

This notebook runs a resource-light LongMemEval reproduction pipeline on Kaggle:

1. Prepare the LongMemEval source tree.
2. Download or import the cleaned benchmark data.
3. Run memory retrieval with BM25 at session granularity.
4. Run retrieval-augmented generation through an OpenAI-compatible API.
5. Run the official LLM-as-judge QA evaluator and summarize results.

The default run is a smoke test: 20 examples, CPU BM25 retrieval, and API-based generation/evaluation. Set `RUN_FULL=1` or edit `RUN_FULL = True` to evaluate all 500 examples.

## Kaggle requirements

- Internet must be enabled unless you attach the benchmark JSON files as a Kaggle Dataset.
- Add `OPENAI_API_KEY` in Kaggle Add-ons -> Secrets.
- If you use a non-OpenAI router, also add `OPENAI_BASE_URL` as a Kaggle Secret or set it in the configuration cell.
- The notebook never prints the API key and does not pass it as a command-line argument.


In [ ]:
from pathlib import Path
import json
import math
import os
import random
import shutil
import subprocess
import sys
import time
import urllib.request

ROOT = Path('/kaggle/working') if Path('/kaggle').exists() else Path.cwd()
PROJECT_DIR_NAME = os.getenv('PROJECT_DIR_NAME', 'LongMemEval-Experiment')
REPO_URL = os.getenv('LONGMEMEVAL_REPO', 'https://github.com/xiaowu0162/LongMemEval.git')
CHECKOUT_DIR = os.getenv('LONGMEMEVAL_CHECKOUT_DIR', PROJECT_DIR_NAME)

# Choose the benchmark file. Use longmemeval_oracle.json for the cheapest generation sanity checks.
DATASET_NAME = os.getenv('LONGMEMEVAL_DATASET', 'longmemeval_s_cleaned.json')

# Smoke-test defaults. For full reproduction, set RUN_FULL=True or RUN_FULL=1 and use all 500 examples.
RUN_FULL = os.getenv('RUN_FULL', '0') == '1'
N_EXAMPLES = int(os.getenv('N_EXAMPLES', '20'))
RANDOM_SEED = int(os.getenv('RANDOM_SEED', '7'))

# Retrieval configuration for the resource-light baseline.
RETRIEVER = os.getenv('RETRIEVER', 'flat-bm25')
GRANULARITY = os.getenv('GRANULARITY', 'session')
TOPK_CONTEXT = int(os.getenv('TOPK_CONTEXT', '50'))

# OpenAI-compatible reader and evaluator configuration.
# Defaults use OpenAI directly. For routers, set OPENAI_BASE_URL and custom model names below.
GEN_MODEL_NAME = os.getenv('GEN_MODEL_NAME', 'gpt-4o-mini-2024-07-18')
GEN_MODEL_ALIAS = os.getenv('GEN_MODEL_ALIAS', 'gpt-4o-mini')
METRIC_MODEL_SHORT = os.getenv('METRIC_MODEL_SHORT', 'gpt-4o-mini')
METRIC_MODEL_NAME = os.getenv('METRIC_MODEL_NAME', GEN_MODEL_NAME)
MODEL_MAX_LENGTH = int(os.getenv('MODEL_MAX_LENGTH', '128000'))
HISTORY_FORMAT = os.getenv('HISTORY_FORMAT', 'json')
USERONLY = os.getenv('USERONLY', 'false')
OPENAI_DEFAULT_HEADERS = os.getenv('OPENAI_DEFAULT_HEADERS', '{}')

print('Working root:', ROOT)
print('Dataset:', DATASET_NAME)
print('Run full benchmark:', RUN_FULL, '| N_EXAMPLES:', N_EXAMPLES)
print('Retriever:', RETRIEVER, '| Granularity:', GRANULARITY)
print('Generation model:', GEN_MODEL_NAME)
print('Metric model alias:', METRIC_MODEL_SHORT, '| metric model:', METRIC_MODEL_NAME)


## Install lightweight dependencies

The full project requirements include vLLM and heavier CUDA packages. For this BM25 + API pipeline, these packages are enough. `httpx==0.27.2` is pinned for compatibility with `openai==1.35.1`.


In [ ]:
%pip install -q openai==1.35.1 httpx==0.27.2 backoff==2.2.1 rank-bm25==0.2.2 tiktoken==0.7.0 sentence-transformers==2.7.0 scikit-learn numpy==1.26.4


In [ ]:
import httpx
import openai
print('openai version:', openai.__version__)
print('httpx version:', httpx.__version__)
assert tuple(map(int, httpx.__version__.split('.')[:2])) < (0, 28), 'httpx must be < 0.28 for openai==1.35.1'


## Load secrets and helper functions

Secrets are read from environment variables first, then from Kaggle Secrets. The API key is only stored in the process environment and is not passed to subprocess command lines.


In [ ]:
def run_cmd(cmd, cwd=None, env=None, check=True):
    shown = [str(x) for x in cmd]
    print('$', ' '.join(shown))
    return subprocess.run(cmd, cwd=str(cwd) if cwd else None, env=env, check=check)


def load_secret(name):
    value = os.getenv(name)
    if value:
        return value
    try:
        from kaggle_secrets import UserSecretsClient
        return UserSecretsClient().get_secret(name)
    except Exception:
        return ''


OPENAI_API_KEY = load_secret('OPENAI_API_KEY')
OPENAI_ORGANIZATION = load_secret('OPENAI_ORGANIZATION')
OPENAI_BASE_URL = load_secret('OPENAI_BASE_URL') or os.getenv('OPENAI_BASE_URL', '')

if OPENAI_API_KEY:
    os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
if OPENAI_ORGANIZATION:
    os.environ['OPENAI_ORGANIZATION'] = OPENAI_ORGANIZATION
if OPENAI_BASE_URL:
    os.environ['OPENAI_BASE_URL'] = OPENAI_BASE_URL
os.environ['OPENAI_DEFAULT_HEADERS'] = OPENAI_DEFAULT_HEADERS
os.environ['TOKENIZER_BACKEND'] = os.getenv('TOKENIZER_BACKEND', 'openai')
os.environ['MODEL_MAX_LENGTH'] = str(MODEL_MAX_LENGTH)
os.environ['METRIC_MODEL_NAME'] = METRIC_MODEL_NAME

print('OPENAI_API_KEY configured:', bool(OPENAI_API_KEY))
print('OPENAI_ORGANIZATION configured:', bool(OPENAI_ORGANIZATION))
print('OPENAI_BASE_URL configured:', bool(OPENAI_BASE_URL))
print('MODEL_MAX_LENGTH:', MODEL_MAX_LENGTH)
print('TOKENIZER_BACKEND:', os.environ['TOKENIZER_BACKEND'])


## Prepare the source tree

When the notebook is run from a repository checkout, it uses that checkout. Otherwise it clones `LONGMEMEVAL_REPO` into `/kaggle/working`. Set `LONGMEMEVAL_REPO` to your fork if you want Kaggle to run this exact project instead of the upstream reference repository.


In [ ]:
def looks_like_longmemeval_repo(path):
    path = Path(path)
    return (path / 'src' / 'retrieval' / 'run_retrieval.py').exists() and (path / 'src' / 'generation' / 'run_generation.py').exists()


candidate_dirs = [Path.cwd(), ROOT / CHECKOUT_DIR, ROOT / 'LongMemEval', ROOT / PROJECT_DIR_NAME]
REPO_DIR = None
for candidate in candidate_dirs:
    if looks_like_longmemeval_repo(candidate):
        REPO_DIR = candidate.resolve()
        break

if REPO_DIR is None:
    REPO_DIR = (ROOT / CHECKOUT_DIR).resolve()
    if not REPO_DIR.exists():
        run_cmd(['git', 'clone', '--depth', '1', REPO_URL, str(REPO_DIR)])
    if not looks_like_longmemeval_repo(REPO_DIR):
        raise RuntimeError(f'Checkout does not look like a LongMemEval repo: {REPO_DIR}')

print('Using source tree:', REPO_DIR)
print('Top-level files:')
for path in sorted(REPO_DIR.iterdir()):
    if path.name != '.git':
        print(' -', path.name)


## Apply Kaggle compatibility patches

These patches are idempotent. They keep the benchmark logic intact while making the scripts safer for Kaggle: no API key in printed args, optional OpenAI-compatible base URL, custom metric model alias support, and NumPy 2.x compatibility for retrieval metrics.


In [ ]:
def replace_text(path, old, new):
    text = path.read_text(encoding='utf-8')
    if old in text:
        path.write_text(text.replace(old, new), encoding='utf-8')
        return True
    return False


gen_py = REPO_DIR / 'src' / 'generation' / 'run_generation.py'
gen_text = gen_py.read_text(encoding='utf-8')
if 'import os\n' not in gen_text[:150]:
    gen_text = gen_text.replace('import sys\n', 'import sys\nimport os\n')
gen_py.write_text(gen_text, encoding='utf-8')

replace_text(
    gen_py,
    "    if args.openai_organization:\n        openai.organization = args.openai_organization\n",
    "    openai_organization = args.openai_organization or os.getenv('OPENAI_ORGANIZATION')\n    if openai_organization:\n        openai.organization = openai_organization\n",
)

replace_text(
    gen_py,
    "    parser.add_argument('--openai_key', type=str, required=True)\n",
    "    parser.add_argument('--openai_key', type=str, default=None)\n",
)
replace_text(
    gen_py,
    "def check_args(args):\n    print(args)\n",
    "def check_args(args):\n    safe_args = argparse.Namespace(**vars(args))\n    if safe_args.openai_key:\n        safe_args.openai_key = '***'\n    if safe_args.openai_organization:\n        safe_args.openai_organization = '***'\n    print(safe_args)\n",
)
replace_text(
    gen_py,
    "    client = OpenAI(\n        api_key=args.openai_key,\n        base_url=args.openai_base_url,\n    )",
    "    openai_key = args.openai_key or os.getenv('OPENAI_API_KEY')\n    if not openai_key:\n        raise RuntimeError('OPENAI_API_KEY is required. Set it in the environment or pass --openai_key.')\n    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    client = OpenAI(\n        api_key=openai_key,\n        base_url=args.openai_base_url,\n        default_headers=default_headers,\n    )",
)
replace_text(
    gen_py,
    "    model_max_length = model2maxlength[args.model_name]\n",
    "    model_max_length = model2maxlength.get(args.model_name, int(os.getenv('MODEL_MAX_LENGTH', '128000')))\n",
)
replace_text(
    gen_py,
    "    if 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
    "    if os.getenv('TOKENIZER_BACKEND', '').lower() == 'openai' or 'gpt-4' in args.model_name.lower()  or 'gpt-3.5' in args.model_name.lower():\n        tokenizer = tiktoken.get_encoding('o200k_base')\n        tokenizer_backend = 'openai'\n    else:\n        tokenizer = AutoTokenizer.from_pretrained(args.model_name)\n        tokenizer_backend = 'huggingface'\n",
)
replace_text(
    gen_py,
    "            total_prompt_tokens += completion.usage.prompt_tokens\n            total_completion_tokens += completion.usage.completion_tokens\n",
    "            usage = getattr(completion, 'usage', None)\n            total_prompt_tokens += (getattr(usage, 'prompt_tokens', 0) or 0)\n            total_completion_tokens += (getattr(usage, 'completion_tokens', 0) or 0)\n",
)

eval_py = REPO_DIR / 'src' / 'evaluation' / 'evaluate_qa.py'
if "METRIC_MODEL_NAME" not in eval_py.read_text(encoding='utf-8'):
    replace_text(
        eval_py,
        "    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
        "    if metric_model_short not in model_zoo and os.getenv('METRIC_MODEL_NAME'):\n        model_zoo[metric_model_short] = (os.getenv('METRIC_MODEL_NAME'), 'openai')\n    if metric_model_short not in model_zoo:\n        print('Requested metric model is not supported:', metric_model_short)\n        exit()\n",
    )
replace_text(
    eval_py,
    "        openai_api_base = None\n",
    "        openai_api_base = os.getenv('OPENAI_BASE_URL') or None\n        if not openai_api_key:\n            raise RuntimeError('OPENAI_API_KEY is required for OpenAI-compatible evaluation models.')\n",
)
replace_text(
    eval_py,
    "    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n    )",
    "    default_headers = json.loads(os.getenv('OPENAI_DEFAULT_HEADERS', '{}'))\n    metric_client = OpenAI(\n        api_key=openai_api_key,\n        base_url=openai_api_base,\n        default_headers=default_headers,\n    )",
)

eval_utils_py = REPO_DIR / 'src' / 'retrieval' / 'eval_utils.py'
replace_text(eval_utils_py, 'np.asfarray(relevances)[:k]', 'np.asarray(relevances, dtype=float)[:k]')

print('Compatibility patches applied or already present.')


## Fetch benchmark data

The notebook first searches `/kaggle/input` for `DATASET_NAME`. If the file is not attached as a Kaggle Dataset, it downloads the official cleaned benchmark file from Hugging Face.


In [ ]:
DATA_DIR = REPO_DIR / 'data'
DATA_DIR.mkdir(parents=True, exist_ok=True)
target_file = DATA_DIR / DATASET_NAME

if not target_file.exists():
    candidates = list(Path('/kaggle/input').rglob(DATASET_NAME)) if Path('/kaggle/input').exists() else []
    if candidates:
        print('Copying dataset from Kaggle input:', candidates[0])
        shutil.copy2(candidates[0], target_file)
    else:
        url = f'https://huggingface.co/datasets/xiaowu0162/longmemeval-cleaned/resolve/main/{DATASET_NAME}'
        print('Downloading:', url)
        urllib.request.urlretrieve(url, target_file)
else:
    print('Dataset already exists:', target_file)

data = json.loads(target_file.read_text(encoding='utf-8'))
print('Loaded examples:', len(data))
print('First example keys:', sorted(data[0].keys()))
counts = {}
for row in data:
    counts[row['question_type']] = counts.get(row['question_type'], 0) + 1
print('Question type counts:')
print(json.dumps(counts, indent=2))


## Build a reproducible sample file

A sample keeps API spend controlled. To reproduce full benchmark metrics, set `RUN_FULL=True` in the configuration cell or define Kaggle environment variable `RUN_FULL=1`.


In [ ]:
if RUN_FULL:
    work_file = target_file
    work_data = data
else:
    rng = random.Random(RANDOM_SEED)
    work_data = data.copy()
    rng.shuffle(work_data)
    work_data = work_data[:N_EXAMPLES]
    sample_name = f'{target_file.stem}_sample{len(work_data)}_seed{RANDOM_SEED}.json'
    work_file = DATA_DIR / sample_name
    work_file.write_text(json.dumps(work_data, ensure_ascii=False), encoding='utf-8')

print('Active benchmark file:', work_file)
print('Active examples:', len(work_data))


## Step 1: memory retrieval

This runs the repository retrieval code and writes a JSONL retrieval log containing `retrieval_results`. The default is `flat-bm25` over sessions, which is CPU-friendly and mirrors the baseline memory retrieval stage.


In [ ]:
retrieval_out_dir = REPO_DIR / 'retrieval_logs' / RETRIEVER / GRANULARITY
retrieval_out_dir.mkdir(parents=True, exist_ok=True)
env = os.environ.copy()
env['PYTHONPATH'] = str(REPO_DIR) + os.pathsep + env.get('PYTHONPATH', '')

cmd = [
    sys.executable, 'run_retrieval.py',
    '--in_file', str(work_file),
    '--retriever', RETRIEVER,
    '--granularity', GRANULARITY,
    '--index_expansion_method', 'none',
    '--index_expansion_result_join_mode', 'none',
    '--index_expansion_result_cache', 'none',
    '--out_dir', str(retrieval_out_dir),
    '--outfile_prefix', work_file.name,
    '--cache_dir', str(REPO_DIR / 'model_cache'),
]
run_cmd(cmd, cwd=REPO_DIR / 'src' / 'retrieval', env=env)

retrieval_log = retrieval_out_dir / f'{work_file.name}_retrievallog_{GRANULARITY}_{RETRIEVER}'
print('Retrieval log:', retrieval_log)
print('Exists:', retrieval_log.exists(), '| Size MB:', round(retrieval_log.stat().st_size / 1e6, 2) if retrieval_log.exists() else None)


In [ ]:
run_cmd([sys.executable, 'src/evaluation/print_retrieval_metrics.py', str(retrieval_log)], cwd=REPO_DIR, env=env)


## Step 2: retrieval-augmented generation

This stage reads the retrieval log and calls an OpenAI-compatible chat completions API. The API key is read by `run_generation.py` from `OPENAI_API_KEY`, so it is not printed in notebook output.


In [ ]:
if not OPENAI_API_KEY:
    raise RuntimeError('Set Kaggle Secret OPENAI_API_KEY before running generation/evaluation cells.')

run_id = time.strftime('%Y%m%d-%H%M%S')
generation_out_dir = REPO_DIR / 'generation_logs' / f'{RETRIEVER}-{GRANULARITY}' / GEN_MODEL_ALIAS / 'con'
generation_out_dir.mkdir(parents=True, exist_ok=True)

if RETRIEVER == 'oracle':
    retriever_type = f'oracle-{GRANULARITY}'
else:
    retriever_type = f'flat-{GRANULARITY}'

suffix = f'_{run_id}_kaggle'
cmd = [
    sys.executable, 'run_generation.py',
    '--in_file', str(retrieval_log),
    '--out_dir', str(generation_out_dir),
    '--out_file_suffix', suffix,
    '--model_name', GEN_MODEL_NAME,
    '--model_alias', GEN_MODEL_ALIAS,
    '--retriever_type', retriever_type,
    '--merge_key_expansion_into_value', 'none',
    '--topk_context', str(TOPK_CONTEXT),
    '--history_format', HISTORY_FORMAT,
    '--useronly', USERONLY,
    '--cot', 'true',
    '--con', 'false',
]
if OPENAI_BASE_URL:
    cmd.extend(['--openai_base_url', OPENAI_BASE_URL])

run_cmd(cmd, cwd=REPO_DIR / 'src' / 'generation', env=env)

hyp_files = sorted(generation_out_dir.glob(f'*{suffix}'), key=lambda p: p.stat().st_mtime)
if not hyp_files:
    raise FileNotFoundError(f'No generation output found with suffix {suffix}')
hyp_file = hyp_files[-1]
print('Hypothesis file:', hyp_file)
print('Lines:', sum(1 for _ in hyp_file.open(encoding='utf-8')))


## Step 3: official QA evaluation

The official evaluator asks a metric LLM whether each generated answer is correct. For paper-comparable judging, use `METRIC_MODEL_SHORT='gpt-4o'` with an OpenAI endpoint/key. For routers, set `METRIC_MODEL_SHORT` to any alias and `METRIC_MODEL_NAME` to the router model name.


In [ ]:
cmd = [sys.executable, 'evaluate_qa.py', METRIC_MODEL_SHORT, str(hyp_file), str(work_file)]
run_cmd(cmd, cwd=REPO_DIR / 'src' / 'evaluation', env=env)

eval_file = Path(str(hyp_file) + f'.eval-results-{METRIC_MODEL_SHORT}')
print('Evaluation log:', eval_file)
print('Exists:', eval_file.exists())


## Aggregate QA and retrieval metrics

The QA summary supports any evaluator alias. Retrieval metrics below match the reporting rule in `run_retrieval.py`: skip abstention items and items without user-side target labels.


In [ ]:
eval_rows = [json.loads(line) for line in eval_file.read_text(encoding='utf-8').splitlines() if line.strip()]
ref_rows = {row['question_id']: row for row in json.loads(work_file.read_text(encoding='utf-8'))}
retrieval_rows = [json.loads(line) for line in retrieval_log.read_text(encoding='utf-8').splitlines() if line.strip()]
retrieval_by_id = {row['question_id']: row for row in retrieval_rows}

def has_user_side_target(row):
    return any(
        ('has_answer' in turn) and bool(turn['has_answer'])
        for session in row.get('haystack_sessions', [])
        for turn in session
        if turn.get('role') == 'user'
    )

type_to_scores = {}
abstention_scores = []
question_records = []
for row in eval_rows:
    qid = row['question_id']
    ref = ref_rows[qid]
    score = 1 if row['autoeval_label']['label'] else 0
    qtype = ref['question_type']
    type_to_scores.setdefault(qtype, []).append(score)
    if '_abs' in qid:
        abstention_scores.append(score)
    rmetrics = retrieval_by_id.get(qid, {}).get('retrieval_results', {}).get('metrics', {}).get('session', {})
    question_records.append({
        'question_id': qid,
        'question_type': qtype,
        'abstention': '_abs' in qid,
        'correct': bool(score),
        'session_recall_all@5': rmetrics.get('recall_all@5'),
        'session_ndcg_any@5': rmetrics.get('ndcg_any@5'),
        'question': ref['question'],
        'answer': ref['answer'],
        'hypothesis': row.get('hypothesis', ''),
    })

all_scores = [s for scores in type_to_scores.values() for s in scores]
task_scores = [sum(scores) / len(scores) for scores in type_to_scores.values() if scores]

print('Evaluation model:', eval_rows[0]['autoeval_label']['model'] if eval_rows else None)
print('Overall accuracy:', round(sum(all_scores) / len(all_scores), 4) if all_scores else None)
print('Task-averaged accuracy:', round(sum(task_scores) / len(task_scores), 4) if task_scores else None)
print('Abstention accuracy:', round(sum(abstention_scores) / len(abstention_scores), 4) if abstention_scores else None, f'({len(abstention_scores)})')
print()
print('By question type:')
for qtype, scores in sorted(type_to_scores.items()):
    print(f'  {qtype}: {sum(scores) / len(scores):.4f} ({len(scores)})')

retrieval_metric_records = []
for granularity in ['session', 'turn']:
    metric_names = sorted({
        name
        for row in retrieval_rows
        for name in row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).keys()
    })
    for metric in metric_names:
        values = []
        for row in retrieval_rows:
            if '_abs' in row['question_id'] or not has_user_side_target(row):
                continue
            value = row.get('retrieval_results', {}).get('metrics', {}).get(granularity, {}).get(metric)
            if value is not None and not (isinstance(value, float) and math.isnan(value)):
                values.append(value)
        if values:
            retrieval_metric_records.append({
                'granularity': granularity,
                'metric': metric,
                'mean': sum(values) / len(values),
                'n': len(values),
            })


In [ ]:
import pandas as pd
pd.set_option('display.max_colwidth', 140)
pd.set_option('display.width', 160)

question_df = pd.DataFrame(question_records)
type_df = (
    question_df.groupby('question_type', dropna=False)
    .agg(n=('correct', 'size'), accuracy=('correct', 'mean'), failures=('correct', lambda s: int((~s).sum())))
    .reset_index()
    .sort_values(['accuracy', 'n'], ascending=[True, False])
)
retrieval_df = pd.DataFrame(retrieval_metric_records)
summary_df = pd.DataFrame([
    {'metric': 'reference_file', 'value': str(work_file)},
    {'metric': 'retrieval_log', 'value': str(retrieval_log)},
    {'metric': 'hypothesis_file', 'value': str(hyp_file)},
    {'metric': 'evaluation_log', 'value': str(eval_file)},
    {'metric': 'evaluation_model', 'value': eval_rows[0]['autoeval_label']['model'] if eval_rows else None},
    {'metric': 'examples_evaluated', 'value': len(question_df)},
    {'metric': 'overall_accuracy', 'value': round(float(question_df['correct'].mean()), 4)},
    {'metric': 'task_averaged_accuracy', 'value': round(float(type_df['accuracy'].mean()), 4)},
    {'metric': 'abstention_accuracy', 'value': None if question_df[question_df['abstention']].empty else round(float(question_df[question_df['abstention']]['correct'].mean()), 4)},
    {'metric': 'num_failures', 'value': int((~question_df['correct']).sum())},
])

display(summary_df)
display(type_df.assign(accuracy=lambda df: df['accuracy'].round(4)))
if not retrieval_df.empty:
    display(
        retrieval_df.pivot_table(index=['granularity', 'metric'], values='mean')
        .reset_index()
        .assign(mean=lambda df: df['mean'].round(4))
    )
else:
    print('No retrieval metrics found.')


In [ ]:
question_overview_df = question_df.copy()
question_overview_df['correct'] = question_overview_df['correct'].map({True: 'yes', False: 'no'})
question_overview_df['hypothesis_preview'] = question_overview_df['hypothesis'].str.replace(chr(10), ' ', regex=False).str.slice(0, 220)
question_overview_df = question_overview_df[
    ['correct', 'question_type', 'abstention', 'session_recall_all@5', 'session_ndcg_any@5', 'question_id', 'question', 'answer', 'hypothesis_preview']
].sort_values(['correct', 'question_type', 'question_id'])

display(question_overview_df)


In [ ]:
failed_df = question_df[~question_df['correct']].copy()
if failed_df.empty:
    print('No failed examples in this evaluation run.')
else:
    failed_df['hypothesis_preview'] = failed_df['hypothesis'].str.replace(chr(10), ' ', regex=False).str.slice(0, 600)
    display(failed_df[['question_id', 'question_type', 'question', 'answer', 'hypothesis_preview']])


## Optional: long-context baseline

This baseline provides the full recent history to the reader. It is expensive on `longmemeval_s_cleaned.json` and not practical for `longmemeval_m_cleaned.json` with normal Kaggle limits. Enable it only after the smoke test succeeds.


In [ ]:
RUN_LONG_CONTEXT_BASELINE = False

if RUN_LONG_CONTEXT_BASELINE:
    full_out_dir = REPO_DIR / 'generation_logs' / 'full-history-session' / GEN_MODEL_ALIAS / 'con'
    full_out_dir.mkdir(parents=True, exist_ok=True)
    full_suffix = f'_{run_id}_kaggle_fullhistory'
    cmd = [
        sys.executable, 'run_generation.py',
        '--in_file', str(work_file),
        '--out_dir', str(full_out_dir),
        '--out_file_suffix', full_suffix,
        '--model_name', GEN_MODEL_NAME,
        '--model_alias', GEN_MODEL_ALIAS,
        '--retriever_type', 'orig-session',
        '--merge_key_expansion_into_value', 'none',
        '--topk_context', '1000',
        '--history_format', HISTORY_FORMAT,
        '--useronly', USERONLY,
        '--cot', 'true',
        '--con', 'false',
    ]
    if OPENAI_BASE_URL:
        cmd.extend(['--openai_base_url', OPENAI_BASE_URL])
    run_cmd(cmd, cwd=REPO_DIR / 'src' / 'generation', env=env)
else:
    print('Skipped. Set RUN_LONG_CONTEXT_BASELINE=True to run this optional baseline.')


## Optional: package outputs

Run this cell to create a downloadable archive under Kaggle output.


In [ ]:
archive = ROOT / 'longmemeval_repro_outputs.tar.gz'
run_cmd(['tar', '-czf', str(archive), 'retrieval_logs', 'generation_logs'], cwd=REPO_DIR, env=env, check=False)
print('Archive:', archive)
